In [2]:
import sys
sys.path.append("..")
from src.db import query_cached, append_note, get_schema
from src.cleaning import detect_contaminated_devices, filter_non_laptop_devices, normalize_laptop_weight
import pandas as pd
from datetime import datetime

df_analysis = pd.read_csv("../data/processed/analysis_base.csv")

In [3]:
### [Giai đoạn 5 - Trang 1] Overview stats cho Manager
overview_query = """
SELECT
    COUNT(DISTINCT user_psuedo_id) AS total_active_users,
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(*) AS total_events
FROM laplaptech.user_event_tracking
"""
df_overview = query_cached(overview_query, cache_dir="../data/raw", force_refresh=True)
display(df_overview)

# Phân bố "người dùng" theo phân khúc máy: dùng interest_score/pageview_count (mức độ chú ý
# thực tế của người dùng), KHÔNG chỉ đếm số lượng laptop trong từng phân khúc.
def segment_label(row):
    if row["is_gaming_laptop"] == True:
        return "Gaming"
    if row["is_workstation"] == True:
        return "Workstation"
    return "Office"

df_analysis["segment"] = df_analysis.apply(segment_label, axis=1)
df_segment_attention = df_analysis.groupby("segment").agg(
    laptop_count=("id", "count"),
    total_pageview=("pageview_count", "sum"),
    total_interest_score=("interest_score", "sum"),
).reset_index()
display(df_segment_attention)

df_overview.to_csv("../data/processed/dashboard_overview_kpi.csv", index=False)
df_segment_attention.to_csv("../data/processed/dashboard_segment_attention.csv", index=False)
print("Đã lưu: dashboard_overview_kpi.csv, dashboard_segment_attention.csv")
print("Top laptop dùng trực tiếp analysis_base.csv (sort theo interest_score trong Power BI), không cần file riêng.")

,total_active_users,total_sessions,total_events
0,34783,87292,773851


,segment,laptop_count,total_pageview,total_interest_score
0,Gaming,47,30502,46264
1,Office,86,71965,113301
2,Workstation,2,1818,2694


Đã lưu: dashboard_overview_kpi.csv, dashboard_segment_attention.csv
Top laptop dùng trực tiếp analysis_base.csv (sort theo interest_score trong Power BI), không cần file riêng.


In [4]:
### [Giai đoạn 5 - Trang 2] CPU/GPU battery ratio — GIỮ NGUYÊN theo kế hoạch gốc nhưng flag rõ độ thưa
cpu_gpu_query = """
SELECT
    c.name AS cpu_name,
    b.geekbench_6_cpu_multi_core_plugged_in,
    b.geekbench_6_cpu_multi_core_battery,
    b.laptop_model_id
FROM laplaptech.laptop_benchmark_result b
INNER JOIN laplaptech.laptop_model m ON b.laptop_model_id = m.id
INNER JOIN laplaptech.cpu_model c ON m.cpu_model_id = c.id
WHERE b.geekbench_6_cpu_multi_core_plugged_in IS NOT NULL
  AND b.geekbench_6_cpu_multi_core_battery IS NOT NULL
"""
df_cpu_gpu = query_cached(cpu_gpu_query, cache_dir="../data/raw", force_refresh=True)
df_cpu_gpu["cpu_battery_ratio"] = (
    df_cpu_gpu["geekbench_6_cpu_multi_core_battery"] / df_cpu_gpu["geekbench_6_cpu_multi_core_plugged_in"]
)
print(f"Số dòng có đủ dữ liệu để tính CPU_Battery_Ratio: {len(df_cpu_gpu)}")
display(df_cpu_gpu)
df_cpu_gpu.to_csv("../data/processed/dashboard_cpu_battery_ratio.csv", index=False)

# Dùng battery_density theo hãng làm chỉ số THAY THẾ có độ phủ tốt hơn cho Trang 2
# (đã dùng cùng logic pivot ở Giai đoạn 3 — Golden Ratio'/H9')
density_by_segment_query = """
SELECT c.name AS cpu_name
FROM laplaptech.cpu_model c
"""  # placeholder, join thật thực hiện ở pandas bên dưới để tái dùng population đã sạch
df_laptop_raw = query_cached("SELECT * FROM laplaptech.laptop_model", cache_dir="../data/raw")
df_cpu_names = query_cached("SELECT id, name FROM laplaptech.cpu_model", cache_dir="../data/raw")
df_laptop_clean = filter_non_laptop_devices(df_laptop_raw)
df_laptop_clean = df_laptop_clean.merge(df_cpu_names, left_on="cpu_model_id", right_on="id",
                                          suffixes=("", "_cpu"))
df_laptop_clean["laptop_weight_g"] = normalize_laptop_weight(df_laptop_clean)
df_laptop_clean["battery_density"] = pd.to_numeric(
    df_laptop_clean["battery_capacity_whr"], errors="coerce"
) / df_laptop_clean["laptop_weight_g"].replace(0, pd.NA)

df_density_by_cpu = df_laptop_clean.groupby("name_cpu")["battery_density"].agg(
    ["count", "mean", "median"]
).reset_index().sort_values("count", ascending=False)
display(df_density_by_cpu)
df_density_by_cpu.to_csv("../data/processed/dashboard_battery_density_by_cpu.csv", index=False)

note_page2 = f"""
## [Giai đoạn 5 - Trang 2] Ghi chú độ khả dụng dữ liệu

CPU_Battery_Ratio (theo kế hoạch gốc): chỉ {len(df_cpu_gpu)} dòng có đủ dữ liệu Geekbench
plugged-in + battery trên toàn bộ laptop_benchmark_result -> KHÔNG đủ để vẽ biểu đồ có ý nghĩa
theo từng CPU/GPU trên Trang 2. Đã xuất dashboard_cpu_battery_ratio.csv nguyên trạng (để minh
bạch, không giấu), nhưng khuyến nghị KHÔNG dùng làm biểu đồ chính.

Thay thế: dashboard_battery_density_by_cpu.csv — battery_density (Whr/gram) trung bình/trung vị
theo từng dòng CPU, tính trên population đã lọc sạch, độ phủ tốt hơn nhiều. Khi dựng Trang 2,
dùng file này làm biểu đồ chính, và Scatter Trọng lượng vs Battery Density (đã có sẵn dữ liệu
trong analysis_base.csv) thay cho Scatter Trọng lượng vs Thời lượng pin trong kế hoạch gốc,
vì office_battery_result_minutes cũng chỉ có 1/135 dòng (đã pivot từ Giai đoạn 3).
"""
append_note(note_page2)
print(note_page2)

Số dòng có đủ dữ liệu để tính CPU_Battery_Ratio: 1


,cpu_name,geekbench_6_cpu_multi_core_plugged_in,geekbench_6_cpu_multi_core_battery,laptop_model_id,cpu_battery_ratio
0,Qualcomm Snapdragon X2 Elite Extreme X2E 96 100,22252.0,21259.0,157,0.955375


,name_cpu,count,mean,median
35,Intel Core Ultra 7 258V,9,0.054076,0.056062
7,AMD Ryzen AI 7 350,6,0.051640,0.051000
30,Intel Core Ultra 7 155H,5,0.045290,0.048993
11,AMD Ryzen AI 9 HX 370,5,0.052464,0.051714
29,Intel Core Ultra 5 226V,4,0.050685,0.051546
...,...,...,...,...
59,Intel Core i9-14900HX,1,0.043975,0.043975
66,Qualcomm Snapdragon X2 Elite X2E 88 100,1,0.070707,0.070707
63,Qualcomm Snapdragon X Plus X1P42100,1,0.038514,0.038514
18,Apple M3 Ultra,0,NaN,NaN



## [Giai đoạn 5 - Trang 2] Ghi chú độ khả dụng dữ liệu

CPU_Battery_Ratio (theo kế hoạch gốc): chỉ 1 dòng có đủ dữ liệu Geekbench
plugged-in + battery trên toàn bộ laptop_benchmark_result -> KHÔNG đủ để vẽ biểu đồ có ý nghĩa
theo từng CPU/GPU trên Trang 2. Đã xuất dashboard_cpu_battery_ratio.csv nguyên trạng (để minh
bạch, không giấu), nhưng khuyến nghị KHÔNG dùng làm biểu đồ chính.

Thay thế: dashboard_battery_density_by_cpu.csv — battery_density (Whr/gram) trung bình/trung vị
theo từng dòng CPU, tính trên population đã lọc sạch, độ phủ tốt hơn nhiều. Khi dựng Trang 2,
dùng file này làm biểu đồ chính, và Scatter Trọng lượng vs Battery Density (đã có sẵn dữ liệu
trong analysis_base.csv) thay cho Scatter Trọng lượng vs Thời lượng pin trong kế hoạch gốc,
vì office_battery_result_minutes cũng chỉ có 1/135 dòng (đã pivot từ Giai đoạn 3).



In [5]:
### [Giai đoạn 5 - Trang 3] Funnel: Tìm kiếm -> Xem chi tiết -> Chọn So sánh
# Endpoint đã đổi so với kế hoạch gốc: không có event "xem video review" trong dữ liệu
# (đã xác nhận ở Giai đoạn 3), nên funnel dừng ở "Chọn So sánh".
funnel_query = """
SELECT
    'Tim kiem' AS stage, COUNT(DISTINCT session_id) AS session_count, 1 AS stage_order
FROM laplaptech.user_event_tracking WHERE event_name = 'search_for_device'
UNION ALL
SELECT
    'Xem chi tiet', COUNT(DISTINCT session_id), 2
FROM laplaptech.user_event_tracking
WHERE event_name = 'pageview' AND JSONExtractString(event_data, 'page_name') = 'DeviceDetail'
UNION ALL
SELECT
    'Chon So sanh', COUNT(DISTINCT session_id), 3
FROM laplaptech.user_event_tracking WHERE event_name = 'select_device_for_comparison'
"""
df_funnel = query_cached(funnel_query, cache_dir="../data/raw", force_refresh=True)
df_funnel = df_funnel.sort_values("stage_order")
display(df_funnel)
df_funnel.to_csv("../data/processed/dashboard_funnel_summary.csv", index=False)

note_page3 = """
## [Giai đoạn 5 - Trang 3] Ghi chú funnel

Funnel đếm số session PHÂN BIỆT có xuất hiện event ở mỗi bước (không bắt buộc thứ tự thời gian
trong cùng session) — là ước lượng mức độ tham gia từng bước, không phải phễu chuyển đổi
tuyệt đối theo trình tự. Endpoint đổi từ "click video review" (kế hoạch gốc) sang
"select_device_for_comparison", do không có dữ liệu tracking cho hành vi xem video.
"""
append_note(note_page3)

,stage,session_count,stage_order
0,Tim kiem,37493,1
2,Xem chi tiet,44134,2
1,Chon So sanh,8477,3


In [6]:
### [Giai đoạn 5 - Data Quality Monitor] Snapshot % null + anomaly, có timestamp để theo dõi theo thời gian
snapshot_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

cols_to_check = ["battery_capacity_whr", "laptop_weight", "cpu_tdp", "gpu_tdp",
                  "office_battery_result_minutes", "geekbench_6_cpu_multi_core_plugged_in"]
df_benchmark_raw = query_cached("SELECT * FROM laplaptech.laptop_benchmark_result",
                                  cache_dir="../data/raw")
df_merged_quality = df_laptop_raw.merge(
    df_benchmark_raw, left_on="id", right_on="laptop_model_id", how="left", suffixes=("", "_bm")
)

quality_rows = []
for col in cols_to_check:
    if col in df_merged_quality.columns:
        null_pct = df_merged_quality[col].isna().mean()
        quality_rows.append({"snapshot_time": snapshot_time, "column": col, "null_pct": round(null_pct, 4)})
df_quality_snapshot = pd.DataFrame(quality_rows)
display(df_quality_snapshot)

n_contaminated = len(detect_contaminated_devices(df_laptop_raw))
df_quality_snapshot["contaminated_device_count"] = n_contaminated

# APPEND vào file lịch sử (không ghi đè) để tích lũy dữ liệu theo thời gian cho Power BI vẽ trend
import os
quality_path = "../data/processed/dashboard_data_quality_history.csv"
if os.path.exists(quality_path):
    df_quality_snapshot.to_csv(quality_path, mode="a", header=False, index=False)
else:
    df_quality_snapshot.to_csv(quality_path, mode="w", header=True, index=False)
print(f"Đã append snapshot vào {quality_path}")

,snapshot_time,column,null_pct
0,2026-09-19 20:37:52,battery_capacity_whr,0.0314
1,2026-09-19 20:37:52,laptop_weight,0.0314
2,2026-09-19 20:37:52,cpu_tdp,0.8868
3,2026-09-19 20:37:52,gpu_tdp,0.7862
4,2026-09-19 20:37:52,office_battery_result_minutes,0.9937
5,2026-09-19 20:37:52,geekbench_6_cpu_multi_core_plugged_in,0.9937


Đã append snapshot vào ../data/processed/dashboard_data_quality_history.csv


In [ ]:
### [Giai đoạn 5] Data Dictionary cho Dashboard + quyết định refresh/schema
dashboard_dict = """
## [Giai đoạn 5] Data Dictionary cho Dashboard

| Metric hiển thị | Nguồn file | Công thức/Định nghĩa |
|---|---|---|
| Total Active Users | dashboard_overview_kpi.csv | COUNT(DISTINCT user_psuedo_id) toàn bộ user_event_tracking |
| Total Sessions | dashboard_overview_kpi.csv | COUNT(DISTINCT session_id) |
| Interest Score | analysis_base.csv | 1*pageview(DeviceDetail) + 2*select_device_for_comparison (xem Giai đoạn 3) |
| Battery Density | analysis_base.csv | battery_capacity_whr / laptop_weight_g (đã chuẩn hóa đơn vị) |
| Golden Ratio Group | golden_ratio_laptops.csv | Top 25% battery_density + top 25% nhẹ nhất |
| Funnel Stage | dashboard_funnel_summary.csv | Số session phân biệt có event tương ứng, không bắt buộc thứ tự |
| Data Quality % Null | dashboard_data_quality_history.csv | Tỷ lệ NULL theo cột, snapshot theo thời gian chạy notebook |

## [Giai đoạn 5] Quyết định Data Refresh & Schema Change

**Refresh strategy:** dashboard đọc từ các file CSV tĩnh trong data/processed/, KHÔNG kết nối
trực tiếp ClickHouse. Để cập nhật dữ liệu mới: chạy lại toàn bộ notebook 01 -> 05 theo thứ tự,
sau đó refresh nguồn dữ liệu trong Power BI (Refresh All). Không có lịch tự động — refresh thủ
công trước mỗi lần demo/báo cáo.

**Xử lý khi schema đổi:** notebook 01 (Giai đoạn 0) luôn chạy get_schema() để xác nhận
tên bảng/cột thật trước khi query — nếu ClickHouse đổi schema, các query sẽ lỗi ngay tại
notebook 01/02 thay vì âm thầm ra sai số ở Power BI. detect_contaminated_devices() cũng tự
phát hiện lại mỗi lần chạy (không hardcode ID), nên population sạch tự thích ứng dữ liệu mới.
"""
append_note(dashboard_dict)


## [Giai đoạn 5] Data Dictionary cho Dashboard

| Metric hiển thị | Nguồn file | Công thức/Định nghĩa |
|---|---|---|
| Total Active Users | dashboard_overview_kpi.csv | COUNT(DISTINCT user_psuedo_id) toàn bộ user_event_tracking |
| Total Sessions | dashboard_overview_kpi.csv | COUNT(DISTINCT session_id) |
| Interest Score | analysis_base.csv | 1*pageview(DeviceDetail) + 2*select_device_for_comparison (xem Giai đoạn 3) |
| Battery Density | analysis_base.csv | battery_capacity_whr / laptop_weight_g (đã chuẩn hóa đơn vị) |
| Golden Ratio Group | golden_ratio_laptops.csv | Top 25% battery_density + top 25% nhẹ nhất |
| Funnel Stage | dashboard_funnel_summary.csv | Số session phân biệt có event tương ứng, không bắt buộc thứ tự |
| Data Quality % Null | dashboard_data_quality_history.csv | Tỷ lệ NULL theo cột, snapshot theo thời gian chạy notebook |

## [Giai đoạn 5] Quyết định Data Refresh & Schema Change

**Refresh strategy:** dashboard đọc từ các file CSV tĩnh trong data/processed/, KH

In [ ]:
### Peer-review giả lập

peer_review_note = """
## [Giai đoạn 6 - 6.7] Peer-review giả lập

Tự đọc lại toàn bộ project_notes.md + reports/insight_summary.md với vai trò reviewer, checklist:

1. Insight 1 (Golden Ratio ultrabook): actionable? CÓ (badge/filter cụ thể). Overclaim?
   KHÔNG — đã ghi rõ giới hạn "đo mật độ danh định, không phải hiệu năng thực tế".
2. Insight 2 (Văn phòng > Gaming): actionable? CÓ (rà lại tỷ trọng hiển thị). Overclaim?
   Đã sửa 1 lần (bản đầu lỡ diễn giải theo giả thuyết gốc thay vì chiều thật của dữ liệu) —
   đã đính chính rõ ràng, giữ effect size trung bình (r=0.236) không phóng đại thành "rất mạnh".
3. Insight 3 (2 model gaming pin kém): actionable? CÓ nhưng phạm vi hẹp (2 model cụ thể).
   Overclaim? Đã tự phát hiện và sửa: ban đầu có 5 laptop, sau kiểm tra confound chỉ giữ 2 làm
   trọng tâm, tránh phóng đại số lượng bằng chứng.

Điểm cần lưu ý khi trình bày phỏng vấn: KHÔNG nói "chúng tôi phát hiện X" một cách chắc chắn
tuyệt đối cho bất kỳ insight nào — luôn kèm cỡ mẫu và effect size khi được hỏi sâu, vì cỡ mẫu
population sạch chỉ 135 laptop, không phải dữ liệu lớn.

Kết luận: các insight hiện tại đã ở mức actionable và không overclaim, đạt yêu cầu tự-review.
"""
append_note(peer_review_note)